# Linear Algebra for Data Science 2026 — Lab 2, Core A
# Power iteration and inverse iteration

### <div align="right"> &copy; Rostyslav Hryniv, 2026, after V. Kuchynskyi & R. Hryniv, 2023 </div>

## Completed by:
*   Andrii Polishko
*   Maria Cherkasova

**Marked at the defence:** Core A is worth 2 points; the starred extensions at the end are one of three routes to the extension point (see the brief).

## Why this core

PageRank (Core B), the smoothing step in a graph neural network, and — later in the course — the leading principal component all need **one** eigenpair of a very large matrix that you can only *multiply by*. Power iteration is the algorithm that needs nothing more than $\mathbf{x}\mapsto A\mathbf{x}$. This core makes you implement it, **predict its speed before you run it**, find out where it breaks, and then use *inverse iteration with a shift* to reach the other eigenvalues.

**Uses:** Lecture 3.1 ($A^n = PD^nP^{-1}$, the Jordan asymptotics $n^k|\lambda|^n$, the role of $\kappa(P)$) and Lecture 3.2 (power iteration, inverse iteration). LU factorisation is a Module 0 prerequisite.

**Rules for this notebook**
* Inside `power_method` and `inverse_iteration` do **not** call `np.linalg.eig`, `np.linalg.eigvals`, `scipy.linalg.eig` or anything that calls them. You may use them elsewhere to *check* your results.
* Never call `np.linalg.inv` anywhere in this notebook. If you need $P^{-1}$ applied to something, solve a system.
* Every number you report must be reproduced by `Restart & Run All`.

In [4]:
!pip install numpy scipy matplotlib plotly

In [5]:
%pip install --upgrade nbformat

Note: you may need to restart the kernel to use updated packages.


In [6]:
import sys, numpy as np, scipy, scipy.linalg as sla, matplotlib
import plotly.graph_objects as go
import matplotlib.pyplot as plt
print('python', sys.version.split()[0], '| numpy', np.__version__,
      '| scipy', scipy.__version__, '| matplotlib', matplotlib.__version__)
rng = np.random.default_rng(2026)   # do not change the seed: the defence uses your numbers

python 3.14.0 | numpy 2.5.4 | scipy 1.18.1 | matplotlib 3.11.2


### Two provided helpers

`sin_angle(u, v)` measures how far apart two *directions* are. It ignores length and sign (and, for complex vectors, any unimodular factor), so it is the right error measure for an eigenvector.

`similar_matrix(eigvals, P)` returns $A = P\,\mathrm{diag}(\text{eigvals})\,P^{-1}$ **without** forming $P^{-1}$: from $AP = PD$ we get $P^\top A^\top = (PD)^\top$, which is one linear solve.

In [7]:
def sin_angle(u, v):
    # sine of the angle between span{u} and span{v}; accurate for small angles
    u = u / np.linalg.norm(u)
    v = v / np.linalg.norm(v)
    return np.linalg.norm(u - v * np.vdot(v, u))

def similar_matrix(eigvals, P):
    # A = P diag(eigvals) P^{-1}, computed by one solve instead of an inverse
    PD = P * np.asarray(eigvals)[None, :]          # scales column j of P by eigvals[j]
    return np.linalg.solve(P.T, PD.T).T

def random_orthogonal(n):
    Q, R = np.linalg.qr(rng.standard_normal((n, n)))
    return Q * np.sign(np.diag(R))                  # sign fix: a uniformly random orthogonal Q

---
## 1. Power iteration

### 1.1 The idea, recalled
Let $A$ be a diagonalisable $k\times k$ matrix with eigenvalues ordered so that $|\lambda_1| > |\lambda_2| \ge \dots \ge |\lambda_k|$ (so $\lambda_1$ is real and simple when $A$ is real) and eigenvectors $\mathbf{v}_1,\dots,\mathbf{v}_k$. Write $\mathbf{x}_0 = c_1\mathbf{v}_1 + \dots + c_k\mathbf{v}_k$ with $c_1 \ne 0$. Then
$$
A^n\mathbf{x}_0 = \lambda_1^n\Bigl[c_1\mathbf{v}_1 + c_2\Bigl(\tfrac{\lambda_2}{\lambda_1}\Bigr)^n\mathbf{v}_2 + \dots + c_k\Bigl(\tfrac{\lambda_k}{\lambda_1}\Bigr)^n\mathbf{v}_k\Bigr],
$$
so the direction of $A^n\mathbf{x}_0$ approaches $\mathrm{span}\{\mathbf{v}_1\}$, and the error shrinks by roughly $|\lambda_2/\lambda_1|$ per step. Normalising at each step avoids overflow:
$$
\mathbf{x}_n := \frac{A\mathbf{x}_{n-1}}{\|A\mathbf{x}_{n-1}\|}, \qquad \lambda_1 \approx \mu_n := \mathbf{x}_n^{*} A\,\mathbf{x}_n \quad(\text{the Rayleigh quotient}).
$$

**When to stop.** Do *not* stop when $\|\mathbf{x}_n - \mathbf{x}_{n-1}\|$ is small: if $\lambda_1 < 0$, the vector flips sign at every step and never settles, although its *direction* has converged. Stop instead when the **residual** is small,
$$
\|A\mathbf{x}_n - \mu_n\mathbf{x}_n\| \le \texttt{tol}\cdot\|A\|_F .
$$
(Scaling by $\|A\|_F$, computed once, makes the test independent of units. Do not divide by $\|A\mathbf{x}_n\|$ instead: that breaks down when the eigenvalue you are after is $0$.)
A small residual says that $(\mu_n, \mathbf{x}_n)$ is an exact eigenpair of a matrix very close to $A$ — the most you can ask of any floating-point method.

### 1.2 Implement `power_method`
Return the eigenvalue estimate, the unit eigenvector estimate and a history: the Rayleigh quotients and the residuals at every step. Use `np.vdot` for $\mathbf{x}^*\mathbf{y}$ so that complex input also works.

In [25]:
def power_method(A, x0=None, tol=1e-10, max_iter=1000):
    '''
    Dominant eigenpair of A by power iteration.

    Args:
        A        : (k, k) array
        x0       : starting vector; random (from `rng`) if None
        tol      : stop when ||A x - mu x|| <= tol * ||A||_F
        max_iter : maximum number of matrix-vector products
    Returns:
        mu   : eigenvalue estimate (Rayleigh quotient)
        x    : unit eigenvector estimate
        hist : dict with arrays 'mu' and 'res', one entry per iteration
    '''

    k = A.shape[0]
    frob_norm = sla.norm(A, 'fro')
    stop = tol * frob_norm
    hist = {"mu": [], "res": []}
    x_n = x0 if x0 is not None else rng.standard_normal(size=k)
    A_xn = A @ x_n
    x_star = x_n.T.conj()
    mu_n = x_star @ A_xn

    for _ in range(max_iter):
        x_n = A_xn / sla.norm(A_xn)
        A_xn = A @ x_n
        x_star = x_n.T.conj()
        mu_n = x_star @ A_xn
        part_one = A_xn
        part_two = mu_n * x_n
        res_n = sla.norm(part_one - part_two)
        hist['mu'].append(mu_n)
        hist['res'].append(res_n)

        if res_n <= stop:
            break

    return mu_n, x_n,hist

### 1.3 Test it
The checker compares with `np.linalg.eig`. Its tolerances are matched to `tol`: a residual of $10^{-10}$ cannot give 15 correct digits, so do not tighten them.

The test matrices are chosen on purpose: `A1` has a **negative** dominant eigenvalue, `A2` is symmetric and indefinite, `A3` is non-symmetric with prescribed real eigenvalues.

In [40]:
def check_dominant(A, mu, x, rtol=1e-6):
    # we get eigenvalues and eigenvectors for check
    lam, V = np.linalg.eig(A)
    # getting id of eigenvalue with the larget absolute value
    i = np.argmax(np.abs(lam))
    # Test#1 check if mu is
    assert abs(mu - lam[i]) <= rtol * abs(lam[i]), f'eigenvalue {mu} vs {lam[i]}'
    assert sin_angle(x, V[:, i]) <= rtol, f'eigenvector off by sin(angle) = {sin_angle(x, V[:, i]):.1e}'
    print(f'passed: mu = {complex(mu).real:+.10f}')

A1 = np.array([[0., 1.], [-2., -3.]])
M = rng.standard_normal((6, 6)); A2 = M + M.T
A3 = similar_matrix([-5., 3., 2., 1., 0.5], rng.standard_normal((5, 5)))
for A in (A1, A2, A3):
    mu, x, hist = power_method(A)
    check_dominant(A, mu, x)
    print('   iterations:', len(hist['res']))

passed: mu = -2.0000000010
   iterations: 30
passed: mu = +5.9947898033
   iterations: 58
passed: mu = -5.0000000001
   iterations: 41


**Question 1.3.** Last year's tests used matrices `10*np.random.rand(n, n)`. Explain why even a buggy implementation that cannot cope with a negative dominant eigenvalue would pass all of those tests. *(Hint: the entries are positive; Lecture 3.2, Perron–Frobenius.)*

---
**Your answer here**

---

**Answer**

rand returns elements from 0 to 1, so they are almost always positive, with a very unlikely exception of being 0. By Perron–Frobenius theorem  we know that if all entries of A are positive then $\lambda_1$ with the largest module is also positive AND strictly larger by module for every other eigenvalue. Therefore there would be no such cases that a dominant eigenvalue would be negative.


### 1.4 Predict the speed, then measure it

This is the main experiment of the core. Take $k = 50$, an orthogonal $Q$, and
$$
A_r = Q\,\mathrm{diag}(1,\ r,\ \text{48 values spread evenly in } [-0.3r, 0.3r])\,Q^\top, \qquad r \in \{0.5,\ 0.9,\ 0.99\}.
$$
The dominant eigenvector is $\mathbf{v}_1 = \mathbf{q}_1$, the first column of $Q$.

**(a) Predict first.** For each $r$, how many iterations should it take for $\sin\angle(\mathbf{x}_n, \mathbf{v}_1)$ to fall from about $1$ to $10^{-8}$? Write the formula and the three numbers *before* running anything.

---
**Your answer here**

---

So from eigendecomposition we can see the eigenvalues of A, they are on the diagonal. Dominant always being 1, second dominant is r.

We know that error shrinks roughly by $\lambda_2$ / $\lambda_1$.


for r = 0.5 this will be 0.5. So 0.5^n = 0.00000001. We can find n by solving log_0.5(0.00000001), which gives us gouhly 27 iterations

for r = 0.9 this will be 0.9. So 0.9^n = 0.00000001. We can find n by solving log_0.9(0.00000001), which gives us gouhly 175 iterations

for r = 0.99 this will be 0.99. So 0.99^n = 0.00000001. We can find n by solving log_0.99(0.00000001), which gives us gouhly 1833 iterations


so when the gap between dominant eigenvalues shrunk 10 times from 0.9 to 0.09 the amount of iterations grew roughly 10 times

**(b) Measure.** Write `power_errors(A, v1, lam1, n_steps, x0)` that runs exactly `n_steps` normalised iterations from `x0` and returns two arrays: the eigenvector error $\sin\angle(\mathbf{x}_n,\mathbf{v}_1)$ and the eigenvalue error $|\mu_n - \lambda_1|$. Plot both against $n$ on a log scale for the three values of $r$, with the reference lines $r^n$. Use the **same** `x0` for all runs. Report the measured iteration counts next to your predictions.

In [10]:
def sine_between_vectors(u, v):
    norm_u = u / sla.norm(u)
    norm_v = v / sla.norm(v)

    return sla.norm( norm_u - (norm_u.T @ norm_v) * norm_v)

def power_errors(A, v1, lam1, n_steps, x0):
    # YOUR CODE HERE
    k = A.shape[0]
    x_n = x0 if x0 is not None else rng.random(size=k)
    x_n = x_n / sla.norm(x_n)
    A_xn = A @ x_n
    x_star = x_n.T.conj()
    mu_n = x_star @ A_xn
    sin_between = sine_between_vectors(x_n, v1)
    eigen_err = np.abs(mu_n - lam1)
    sines = []
    erros = []

    for _ in range(n_steps):
        x_n = A_xn / sla.norm(A_xn)
        A_xn = A @ x_n
        x_star = x_n.T.conj()
        mu_n = x_star @ A_xn
        sin_between = sine_between_vectors(x_n, v1)
        eigen_err = np.abs(mu_n - lam1)
        sines.append(sin_between)
        erros.append(eigen_err)

    return np.array(sines), np.array(erros)

k = 50
Q = random_orthogonal(k)
x0 = rng.standard_normal(k)
# YOUR CODE HERE: build A_r for r in (0.5, 0.9, 0.99), run power_errors, plot, report
n = 2000
n_range = np.arange(n)
v1 = Q.T[0]
r_s = (0.5, 0.9, 0.99)
diagonal_values = [np.concatenate(([1, r], np.linspace(-0.3 * r, 0.3 * r, num=k - 2))) for r in r_s]
diagonal_matrixes = [np.diag(diagonal_value) for diagonal_value in diagonal_values]
A_0 = Q @ diagonal_matrixes[0] @ Q.T
A_1 = Q @ diagonal_matrixes[1] @ Q.T
A_2 = Q @ diagonal_matrixes[2] @ Q.T

A_0_vector_error, A_0_value_error = power_errors(A_0, v1, 1, n, x0)
A_1_vector_error, A_1_value_error = power_errors(A_1, v1, 1, n, x0)
A_2_vector_error, A_2_value_error = power_errors(A_2, v1, 1, n, x0)


In [11]:
from plotly.subplots import make_subplots

results = [(A_0_vector_error, A_0_value_error),
           (A_1_vector_error, A_1_value_error),
           (A_2_vector_error, A_2_value_error)]
steps = np.arange(1, n + 1)
colors = ['red', 'blue', 'green']

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("sin(x_n, v_1). Eigenvector errors", "|mu_n - lamda|. Eigenvalue errors"))
for r, (vec_err, val_err), c in zip(r_s, results, colors):
    ref = r ** steps
    for col, err in ((1, vec_err), (2, val_err)):
        fig.add_trace(go.Scatter(x=steps, y=err, name=f"r={r}",
                                 line=dict(color=c), showlegend=(col == 1)), row=1, col=col)
        fig.add_trace(go.Scatter(x=steps, y=ref, name=f"{r}ⁿ",
                                 line=dict(color=c, dash='dash', width=1),
                                 showlegend=(col == 1)), row=1, col=col)

fig.update_yaxes(type="log", range=[-17, 0.5], exponentformat="power")
fig.update_xaxes(title_text="n")
fig.show()

from eigenvalue errors graph we can see that predictions were good.

for r = 0.5 the value reached 10^(-16) at n = 25

for r = 0.9 the value reached 10^(-16) at n = 171

for r = 0.99 the value reached 10^(-16) at n = 1512

**(c) Two observations to explain.**
1. For this symmetric $A_r$ the eigenvalue error falls like $r^{2n}$ — twice as fast as the eigenvector error. You do not need to prove this (Module 4 will), but describe what you see and say why it is good news for PageRank-style uses.
2. Now repeat (b) for $r = 0.9$ only, with $A = P\,\mathrm{diag}(\dots)\,P^{-1}$ built by `similar_matrix`, where $P = U\,\mathrm{diag}(s)\,V^\top$ has orthogonal $U, V$ and singular values $s$ spread logarithmically from $1$ to $10^{-3}$, so that $\kappa(P) = 10^3$. Did the **rate** change? Did anything else change? Relate this to the role of $\kappa(P)$ in Lecture 3.1.

In [24]:
# YOUR CODE HERE for (c)2

k = 50
Q = random_orthogonal(k)
x0 = rng.standard_normal(k)

r = 0.9
# distribution between 1  and 10^(-3)
s = np.logspace(0, -3, k)
# two random orthogonal matrixes
U, V = random_orthogonal(k), random_orthogonal(k)
diagonal_matrix = np.diag(s)
P = U @ diagonal_matrix @ V.T
eigen_vals = np.concatenate(([1, r], np.linspace(-0.3 * r, 0.3 * r, k - 2)))
A = similar_matrix(eigen_vals, P)
v1 = P[:, 0]
A_vector_error, A_value_error = power_errors(A, v1, 1, n, x0)

steps = np.arange(1, n + 1)

# check
print(f'cond = {np.linalg.cond(P)}')
print(f'is v1 an eigen vector of A? {"Yes" if np.isclose(sla.norm(A @ v1 - v1), 0) else "No"} ')

# plots
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=("sin(x_n, v_1). Eigenvector errors", "|mu_n - lamda|. Eigenvalue errors"))

vec_err = A_vector_error
val_err = A_value_error
r = 0.9
c = 'blue'
ref = r ** steps

for col, err in ((1, vec_err), (2, val_err)):
    fig.add_trace(go.Scatter(x=steps, y=err, name=f"r={r}",
                                line=dict(color=c), showlegend=(col == 1)), row=1, col=col)
    fig.add_trace(go.Scatter(x=steps, y=ref, name=f"{r}ⁿ",
                                line=dict(color=c, dash='dash', width=1),
                                showlegend=(col == 1)), row=1, col=col)

fig.update_yaxes(type="log", range=[-17, 0.5], exponentformat="power")
fig.update_xaxes(title_text="n")
fig.show()


cond = 999.9999999999472
is v1 an eigen vector of A? Yes 


---
**Your answer here**

---

The rate is the same (0.9^n), cause it depends only on eigenvalues, which are not changed.

Two things are different
1. The error floor rose from 10^(-15) to 10^(-12). By a 1000, which is proportional to our condition value. The error floor can be calculated by the formula

e_floor ~ |A| * $\varepsilon$  / (1 − r)

2. We cna see that difference between graph of eigenvalue decay and r^n is smaller now, it shrank from r FINISH THIS

### 1.5 Reasons to fail
For each case below: explain *why* the method fails or slows down, give your own $3\times 3$ example, run `power_method` on it, and describe what you see (no convergence? convergence to the wrong thing? very slow convergence?). Look at `hist['res']` and `hist['mu']`, not only at the final output.

1. The dominant eigenvalues of a real matrix are a complex-conjugate pair.
2. $\lambda_1 = -\lambda_2$ (two real eigenvalues of equal modulus).
3. The dominant eigenvalue is defective: a Jordan block. *Hint:* it does converge — at what rate? Compare with the asymptotics $n^k|\lambda|^n$ from Lecture 3.1. Plot the eigenvector error, the eigenvalue error and the residual against $n$; they do **not** fall at the same rate. What does that tell you about trusting a small residual?
4. The starting vector has $c_1 = 0$. First run `power_method` with its defaults on $A = Q\,\mathrm{diag}(2,1,0.5)\,Q^\top$ (random orthogonal $Q$) and $\mathbf{x}_0 = \mathbf{q}_2$: why does it stop at once, and with what answer? Then switch the stopping test off (`tol=0`, `max_iter=120`) and compare two runs: $A = \mathrm{diag}(2, 1, 0.5)$ with $\mathbf{x}_0 = \mathbf{e}_2$, and $QDQ^\top$ with $\mathbf{x}_0 = \mathbf{q}_2$. They behave differently. Explain why, and predict roughly how many iterations the second one needs before it "notices" $\mathbf{v}_1$.

#### 1

In [97]:
# YOUR CODE HERE: one experiment per case
# 1 For this one we can take a rotation with scaling matrix
def case_1():
    A_1 = np.array([[1, -2, 0], [1,  1, 0], [0,  0, 0.5]])
    n = 1000
    n_range = np.arange(1000)
    mu, x, hist = power_method(A_1, max_iter = n)
    hist_mu, hist_res = hist['mu'], hist['res']

    print(f'Eigenvalues by a library function for check:\n{sla.eigvals(A_1)}')
    print(f'Final eigenvalue and eigenvector estimates:\nmu = {mu}\nx={x}')
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=("History of mu", "History of residuals"))
    fig.add_trace(go.Scatter(x=n_range, y=hist_mu), row=1, col=1)
    fig.add_trace(go.Scatter(x=n_range, y=hist_res), row=1, col=2)
    fig.show()

case_1()

Eigenvalues by a library function for check:
[1. +1.41421356j 1. -1.41421356j 0.5+0.j        ]
Final eigenvalue and eigenvector estimates:
mu = 1.2425167270758126
x=[ 0.2505039  -0.96811559  0.        ]


From the graphs we see that approximated eigenvalue oscillates between around 0.5 and 1.5 and residual oscillates between 1 and 2 with no convergence.

Because there is no single dominant eigenvalue- there is no dominant directiopn of x. Also both A and x are real, which makes it impossible to produce imaginary value as a result

### 2

In [ ]:
def case_2():
    A_2 = np.diag([2, -2, 1])
    n = 1000
    n_range = np.arange(1000)
    mu, x, hist = power_method(A_2, max_iter = n)
    hist_mu, hist_res = hist['mu'], hist['res']

    print(f'Eigenvalues by a library function for check:\n{sla.eigvals(A_2)}')
    print(f'Final eigenvalue and eigenvector estimates:\nmu = {mu}\nx={x}')
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=("History of mu", "History of residuals"))
    fig.add_trace(go.Scatter(x=n_range, y=hist_mu), row=1, col=1)
    fig.add_trace(go.Scatter(x=n_range, y=hist_res), row=1, col=2)
    fig.show()

case_2()

Eigenvalues by a library function for check:
[ 2.+0.j -2.+0.j  1.+0.j]


It never converges and goes on a plateau at a value that is not an eigenvalue because we need one value strictly larger to to shrink v_2

#### 3

In [104]:


def case_3():
    A3 = np.array([[2, 1, 0], [0, 2, 0], [0, 0, 1]])
    n = 1000
    n_range = np.arange(1000)
    mu, x, hist = power_method(A3, max_iter = n)
    hist_mu, hist_res = hist['mu'], hist['res']

    print(f'Eigenvalues by a library function for check:\n{sla.eigvals(A3)}')
    print(f'Final eigenvalue and eigenvector estimates:\nmu = {mu}\nx={x}')

    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=("History of mu", "History of residuals"))
    fig.add_trace(go.Scatter(x=n_range, y=hist_mu), row=1, col=1)
    fig.add_trace(go.Scatter(x=n_range, y=hist_res), row=1, col=2)
    fig.update_xaxes(type="log", title_text="n")
    fig.update_yaxes(type="log", exponentformat="power")
    fig.show()

case_3()

Eigenvalues by a library function for check:
[2.+0.j 2.+0.j 1.+0.j]
Final eigenvalue and eigenvector estimates:
mu = 2.001996948237475
x=[9.99998006e-001 1.99695222e-003 1.47879046e-305]


The value converges because there is no other eigenvalue of the same modulus with a different value, but slowly. At 1000 iterations there is still a preatty big difference with expected eigenvalue. This happens because in Jordan case, the wanted and unwanted parts multiplied by the same value, making the wanted part pulling away only by off diagonal part instead of r^n.

#### 4

In [ ]:
def case_4():
    Q = random_orthogonal(3)
    D = np.diag([2, 1, 0.5])
    A4 = Q @ D @ Q.T
    q1, q2 = Q[:, 0], Q[:, 1]
    e2 = np.array([0, 1, 0])
    n = 120
    n_range = np.arange(1, n + 1)

    print(f'Eigenvalues by a library function for check:\n{sla.eigvals(A4)}')

    # Part 1
    mu, x, hist = power_method(A4, x0=q2)
    print(f'\nDefaults, x0 = q2: stopped after {len(hist["mu"])} iteration(s)'
          f'\nmu = {mu}\nx = {x}\nres = {hist["res"][-1]}')

    # Part 2
    mu_d, x_d, hist_d = power_method(D, x0=e2, tol=0, max_iter=n)
    mu_r, x_r, hist_r = power_method(A4, x0=q2, tol=0, max_iter=n)
    hist_mu_d, hist_res_d = np.real(hist_d['mu']), np.abs(hist_d['res'])
    hist_mu_r, hist_res_r = np.real(hist_r['mu']), np.abs(hist_r['res'])

    print(f'\ndiag(2, 1, 0.5), x0 = e2:\nmu = {mu_d}\nx = {x_d}')
    print(f'\nQDQ^T, x0 = q2:\nmu = {mu_r}\nx = {x_r}\n|q1 . x| = {abs(q1 @ x_r)}')

    switch = np.flatnonzero(hist_mu_r > 1.5)
    print(f'\nQDQ^T run: mu first exceeds 1.5 at n = {switch[0] + 1 if switch.size else None}')

    fig = make_subplots(rows=1, cols=2, subplot_titles=("History of mu", "History of residuals"))
    fig.add_trace(go.Scatter(x=n_range, y=hist_mu_d, name="diag, x0 = e2"), row=1, col=1)
    fig.add_trace(go.Scatter(x=n_range, y=hist_mu_r, name="QDQᵀ, x0 = q2"), row=1, col=1)
    fig.add_trace(go.Scatter(x=n_range, y=np.maximum(hist_res_d, 1e-17), name="diag, x0 = e2", showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=n_range, y=np.maximum(hist_res_r, 1e-17), name="QDQᵀ, x0 = q2", showlegend=False), row=1, col=2)
    fig.update_xaxes(title_text="n")
    fig.update_yaxes(type="log", exponentformat="power", row=1, col=2)
    fig.show()

case_4()

Eigenvalues by a library function for check:
[2. +0.j 0.5+0.j 1. +0.j]

Defaults, x0 = q2: stopped after 1 iteration(s)
mu = 1.0000000000000002
x = [-0.30442603 -0.89101833  0.33676568]
res = 3.188872858294072e-16

diag(2, 1, 0.5), x0 = e2:
mu = 1.0
x = [0. 1. 0.]

QDQ^T, x0 = q2:
mu = 2.000000000000001
x = [-0.51914489 -0.14122207 -0.84293826]
|q1 . x| = 1.0000000000000002

QDQ^T run: mu first exceeds 1.5 at n = 53 (prediction ≈ log2(1e16) ≈ 53)


If the start vector has no component along v_1 (c_1 = 0), power iteration has nothing to amplify, so it cannot find the dominant eigenvector.

With x_0 = q_2, the default run stops after one step with μ ~ 1 and x ~ q_2. That is a real eigenpair, so the residual is tiny and the test accepts it, but it is not the dominant one.

With D = diag(2, 1, 0.5) and x_0 = e_2, the arithmetic is exact, so the v₁ component stays exactly 0 and the method never leaves μ = 1.

With QDQ^T and x_0 = q_2, rounding errors create a tiny v_1 component of about 10^(-16). It doubles relative to q_2 every step, so it becomes noticeable after about log(10^16) ~ 53 steps. We measured the switch from μ = 1 to μ = 2 at n ~ 50 to 55. Rounding errors rescue the method here, but only after a delay.

---
## 2. Inverse iteration with a shift

### 2.1 The idea
If $A\mathbf{v} = \lambda\mathbf{v}$ then $(A - \mu I)^{-1}\mathbf{v} = (\lambda - \mu)^{-1}\mathbf{v}$. So the eigenvalue of $A$ **closest to the shift** $\mu$ becomes the *dominant* eigenvalue of $(A-\mu I)^{-1}$, and power iteration on $(A-\mu I)^{-1}$ finds it. The per-step rate is
$$
\frac{|\lambda_* - \mu|}{|\lambda_{**} - \mu|},
$$
where $\lambda_*$ is the closest eigenvalue and $\lambda_{**}$ the second-closest.

You never form $(A-\mu I)^{-1}$. Each step **solves** $(A - \mu I)\mathbf{y} = \mathbf{x}_{n-1}$. Factorise $A - \mu I = LU$ once with `scipy.linalg.lu_factor`, then call `scipy.linalg.lu_solve` at every step.

### 2.2 Implement `inverse_iteration`
Return the eigenvalue **of $A$** (the Rayleigh quotient $\mathbf{x}^*A\mathbf{x}$, not of the inverse), the unit eigenvector and the same history as before, with the same residual stopping rule applied to $A$.

In [ ]:
def inverse_iteration(A, mu, x0=None, tol=1e-10, max_iter=500):
    '''
    Eigenpair of A for the eigenvalue closest to the shift mu.
    One LU factorisation of A - mu I, reused at every step.
    Returns (lam, x, hist) as in power_method.
    '''
    # YOUR CODE HERE
    k = A.shape[0]
    stop = tol * sla.norm(A, 'fro')
    lu_piv = sla.lu_factor(A - mu * np.eye(k))     # factorise ONCE
    hist = {"mu": [], "res": []}

    x_n = x0 if x0 is not None else rng.standard_normal(size=k)
    x_n = x_n / sla.norm(x_n)
    lam = x_n.conj() @ (A @ x_n)

    for _ in range(max_iter):
        y = sla.lu_solve(lu_piv, x_n)
        x_n = y / sla.norm(y)
        A_xn = A @ x_n
        lam = x_n.conj() @ A_xn
        res_n = sla.norm(A_xn - lam * x_n)
        hist['mu'].append(lam)
        hist['res'].append(res_n)

        if res_n <= stop:
            break

    return lam, x_n, hist

**Question 2.2.** State the cost of your implementation in flops, to leading order, for a $k\times k$ matrix and $m$ iterations. Compare with (i) calling `np.linalg.solve(A - mu*I, x)` inside the loop and (ii) forming `inv(A - mu*I)` once and multiplying. Why is (ii) still worse than your version, even though its cost per step is also $O(k^2)$?

---
**Your answer here**

---

For for one LU factorisation ~ (2/3)k^3.

about 4k^2 per iteration: 2k^2 for the triangular solves, 2k^2 for Ax in the Rayleigh quotient and the residual.

so totall is (2/3)k^3 + 4k^2 flops

(i) Calling solve in the loop refactorises every step: ~ (2/3)mk^3, m times more.
(ii) Forming the inverse costs ~ 2k^3, three times the LU factorisation, while the per-step cost is the same 2k^2 as lu_solve

(ii) is worse because it's less accurate, because the inverse of the nearly singular matrix A − μI is not computed in a backward-stable way, when LU solves are. Also it turns sparse matrices into dense ones, requiring much more storage


### 2.3 Find all eigenvalues of a test matrix
Let $D = \mathrm{diag}(0, 1, \dots, 9)$, let $P$ be a random $10\times10$ matrix, and $A = PDP^{-1}$ (use `similar_matrix`).

**(a)** Explain why the diagonal entries of $D$ are the eigenvalues of $A$ and the columns of $P$ are corresponding eigenvectors.

**(b)** With the shifts $\mu_j = j + 0.3$, $j = 0,\dots,9$, find all ten eigenpairs. Check each against the corresponding column of $P$ with `sin_angle`.

**(c)** Predict the per-step rate for $\mu = j + 0.3$ and for $\mu = j + 0.49$, then measure both for one $j$ of your choice from the history. Which eigenvalue does $\mu = j + 0.5$ find, and why is that shift a bad idea?

---
**Your answer here**

---

a) A = PDP^-1;  AP = PDP^-1P = PD
since D is diagonal, each corresponding element j scales column j of P, So Ap_j = d_jp_j. By the conditions of the task, P is invertible, so no columns = 0


In [130]:
D_vals = np.arange(10.)
P10 = rng.standard_normal((10, 10))
A10 = similar_matrix(D_vals, P10)
# YOUR CODE HERE

print('b:')

for j in range(10):
    mu = j + 0.3
    lam, x, hist = inverse_iteration(A10, mu)
    s = sin_angle(x, P10[:, j])
    iterations_amount = len(hist['mu'])

    print(f" j={j}\n mu={mu}\n lam={lam}\n iters={iterations_amount}\n sin={s}\n")

b:
 j=0
 mu=0.3
 lam=-1.1917505131610529e-09
 iters=23
 sin=3.729028058833404e-09

 j=1
 mu=1.3
 lam=0.9999999992546733
 iters=21
 sin=3.5485289754569028e-09

 j=2
 mu=2.3
 lam=2.0000000005418332
 iters=25
 sin=3.6327265082666597e-09

 j=3
 mu=3.3
 lam=2.999999999990445
 iters=21
 sin=5.225092551606155e-09

 j=4
 mu=4.3
 lam=3.999999997749839
 iters=20
 sin=5.755273860299039e-09

 j=5
 mu=5.3
 lam=4.999999996776204
 iters=25
 sin=4.80331375794878e-09

 j=6
 mu=6.3
 lam=5.999999998338276
 iters=23
 sin=3.2479403295148394e-09

 j=7
 mu=7.3
 lam=7.0000000036288235
 iters=23
 sin=6.901018979448811e-09

 j=8
 mu=8.3
 lam=8.000000001825075
 iters=23
 sin=3.5180756912425774e-09

 j=9
 mu=9.3
 lam=9.000000002846605
 iters=13
 sin=5.487301105202673e-09



---
**Your answer here**

---
c) The rate is measured by the fraction of closesest eigenvalue and second closest: $|\lambda* − \mu| / |\lambda** − \mu|$
- so for $\mu = j + 0.3$ the rate is 0.3/0.7 ~ 0.43, to get to 10^(-10) we calculate ln_0.43 (10^(-10)) and get ~ 28

- for $\mu = j + 0.49$ the rate is 0.49/0.51 ~ 0.96, to get to 10^(-10) we calculate ln_0.96 (10^(-10)) and get ~ 565

In [136]:
def measured_rate(res, n_from, n_to):
    idx = np.arange(n_from, n_to)
    slope = np.polyfit(idx, np.log10(res[idx]), 1)[0]

    return 10 ** slope

j = 4
x0 = rng.standard_normal(10)

for offset, max_it in ((0.3, 500), (0.49, 2000)):
    mu = j + offset
    lam, x, hist = inverse_iteration(A10, mu, x0=x0, max_iter=max_it)
    res = np.abs(np.array(hist['res']))
    measured = measured_rate(res, 3, int(0.8 * len(res)))

    print(f" mu={mu}\n measured {measured}\n iters={len(res)}\n lam={np.real(lam)}\n")

 mu=4.3
 measured 0.42924599663385093
 iters=23
 lam=3.9999999976899288

 mu=4.49
 measured 0.961492480597753
 iters=482
 lam=4.000000002836471



this shift is not so good, cause it should be closer to some eignvalue, and not be in the middle

### 2.4 A shift that is *too* good
Take $\mu = 4 + 10^{-12}$. Compute $\kappa(A_{10} - \mu I)$ with `np.linalg.cond`. By everything you learned in Modules 1–2, solving with this matrix should be hopeless. Run `inverse_iteration` anyway and report the number of iterations and the accuracy of the eigenvector. Then explain, in your own words, why the huge condition number does no harm here. *(Hint: the solution $\mathbf{y}$ is badly wrong in length; in which direction is it wrong?)*

In [ ]:
# YOUR CODE HERE
j = 4
mu = j + 1e-12
M = A10 - mu * np.eye(10)
condition_num = np.linalg.cond(M)
print(f"cond(A10 - mu*I) = {condition_num}")

lam, x, hist = inverse_iteration(A10, mu)

sin = sin_angle(x, P10[:, j])
interations_num = len(hist['mu'])
last_residual = hist['res'][-1]

print(f"iterations = {interations_num}\n")
print(f"lam = {lam}\n|lam - 4| = {abs(lam - 4)}\n")
print(f"sin_angle(x, p_4) = {sin}")
print(f"residual = {last_residual}")

cond(A10 - mu*I) = 243783751239789.7
iterations = 1

lam = 4.000000000001151
|lam - 4| = 1.1510792319313623e-12

sin_angle(x, p_4) = 6.002853777806057e-12
residual = 1.4638796676179129e-12


np.complex128(4.999999999999002+0j)

---
**Your answer here**

---

So the condition number is absolutely massive, but it didn’t harm us. 
Because mu is so close to 4, the error from the bad conditioning points almost entirely in the direction of p_4, the eigenvector we want.
So it only makes the length of y wrong, not its direction, but normalising fixes the length

---
## 3. Conclusions
In a few sentences: what did you learn, what was harder than expected, and which result surprised you? If you hit a wall you could not get past, describe it here **and** in your `README` — an explained wall is worth more at the defence than a clean output you cannot explain.

---
**Your answer here**

---

I've learned 
- about finding eigenvalues iteratively instead of analytical approach. 
- Also that convergance rate of power iterations only depends on ratio of most dominant eigenvalues or (distance from the shift to the nearest eigenvalue) / (distance from the shift to the second-nearest eigenvalue). 
- Also small residual doesn't mean that we found the answer, maybe we stuck on wrong plato, or oscillating between values.


I've expected everything to be hard:)
I was surprised last task, where big conditional number didn't affect the result and by graphs in task 1.5


About the wall, it is hard for me to build intuition for these. I think this is where I am hitting the wall

---
# ★ Extensions
Do **at most one** extension for the whole lab (A1, A2 here, or B1 in the Core B notebook). The point is awarded for **a defended finding**: end your extension with a short paragraph headed *Finding* that states what you found and why.

## ★ A1. The QR algorithm: what `np.linalg.eig` builds on
Starting from $A_1 = A$, repeat: factor $A_k = Q_kR_k$, set $A_{k+1} := R_kQ_k$. Under mild assumptions $A_k$ tends to an upper-triangular matrix with the eigenvalues of $A$ on the diagonal.

1. Show that every $A_{k+1}$ is similar to $A_k$ (write $A_{k+1}$ as $Q_k^\top A_k Q_k$), so the eigenvalues never change.
2. Implement the **unshifted** method (`np.linalg.qr` is allowed). Stop when the largest entry below the diagonal is below $10^{-10}\|A\|_F$. Run it on $A = Q\,\mathrm{diag}(1,2,\dots,10)\,Q^\top$. Count the iterations. Predict that count from the ratios $\lambda_{i+1}/\lambda_i$ before you run it. *(Hint: the unshifted method behaves like power iteration on all columns at once.)*
3. Implement a **shifted** version with deflation: on the active $m\times m$ block, use $\mu = a_{mm}$, factor $A_k - \mu I = QR$, set $A_{k+1} = RQ + \mu I$; when $|a_{m,m-1}|$ is negligible, accept $a_{mm}$ and continue with the leading $(m-1)\times(m-1)$ block. Count the total iterations on the same $A$, and record $|a_{m,m-1}|$ step by step for one eigenvalue. How fast does it fall — do the correct digits roughly double, or more?
4. Run the unshifted method on a real matrix whose dominant eigenvalues are a complex pair. What does $A_k$ look like after many steps? Connect this with failure case 1 of §1.5.

In [ ]:
# YOUR CODE HERE

## ★ A2. Stable, yet it grows: non-normal iterated maps
An iterated map $\mathbf{x}_{n+1} = A\mathbf{x}_n$ with spectral radius $\rho(A) < 1$ always tends to $\mathbf{0}$. It need **not** get there monotonically. This matters whenever a linear recurrence is called "stable" because its eigenvalues are inside the unit circle — a linear RNN or an autoregressive model, for example.

1. Take $A = \begin{pmatrix} 0.95 & 10 \\ 0 & 0.9\end{pmatrix}$. Plot $\|A^n\|_2$ for $n = 0,\dots,200$ together with $\rho(A)^n$ and the bound $\kappa(P)\,\rho(A)^n$ from Lecture 3.1 ($P$ the eigenvector matrix; here you may use `np.linalg.eig`). Find the peak of $\|A^n\|_2$ and the $n$ where it happens.
2. Derive $A^n$ in closed form for this triangular $A$ and use it to explain the peak.
3. Repeat for the $20\times20$ bidiagonal matrix with $0.9$ on the diagonal and $1$ on the superdiagonal. How large does $\|A^n\|_2$ get before it decays? Why is $\kappa(P)$ useless as a bound here?

In [150]:
# YOUR CODE HERE

def part1():
    A = np.array([[0.95, 10.], [0., 0.9]])
    n = 200
    n_range = np.arange(n + 1)

    norms = []
    M = np.eye(2)
    for _ in n_range:
        norms.append(sla.norm(M, 2))
        M = A @ M
    norms = np.array(norms)

    eigvals, P = np.linalg.eig(A)
    rho = np.max(np.abs(eigvals))
    kappa = np.linalg.cond(P)

    peak = np.argmax(norms)
    print(f'rho(A) = {rho}, kappa(P) = {kappa:.1f}')
    print(f'Peak of ||A^n||_2 = {norms[peak]:.2f} at n = {peak}')

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=n_range, y=norms, name="‖Aⁿ‖₂"))
    fig.add_trace(go.Scatter(x=n_range, y=rho ** n_range, name="ρ(A)ⁿ", line=dict(dash='dash')))
    fig.add_trace(go.Scatter(x=n_range, y=kappa * rho ** n_range, name="κ(P)·ρ(A)ⁿ", line=dict(dash='dot')))
    fig.update_xaxes(title_text="n")
    fig.update_yaxes(type="log", exponentformat="power")
    fig.show()

part1()

rho(A) = 0.95, kappa(P) = 400.0
Peak of ||A^n||_2 = 51.83 at n = 13


task (2)

**Closed form.** Multiplying $A$ by itself a few times shows the pattern:

$$
A^n = \begin{pmatrix} 0.95^n & 200\,(0.95^n - 0.9^n) \\ 0 & 0.9^n \end{pmatrix}
$$

(the top-right entry is $10 \cdot \frac{0.95^n - 0.9^n}{0.95 - 0.9}$, from a geometric sum).

**Why there is a peak.** The diagonal entries just shrink. The top-right entry is different:

- at $n = 0$ it is $0$, because $0.95^0 - 0.9^0 = 0$;
- then it grows, because $0.9^n$ shrinks faster than $0.95^n$ and the difference is multiplied by $200$;
- later both terms shrink, so it goes back to $0$.

It is largest around $n \approx 13$, where $200\,(0.95^{13} - 0.9^{13}) \approx 51.8$, which matches the peak of $\|A^n\|_2$.

In [151]:
def part3():
    k = 20
    A = 0.9 * np.eye(k) + np.diag(np.ones(k - 1), 1)
    n = 1000
    n_range = np.arange(n + 1)

    norms = []
    M = np.eye(k)
    for _ in n_range:
        norms.append(sla.norm(M, 2))
        M = A @ M
    norms = np.array(norms)

    eigvals, P = np.linalg.eig(A)
    rho = np.max(np.abs(eigvals))
    print(f'rho(A) = {rho}, kappa(P) = {np.linalg.cond(P):.1e}')

    peak = np.argmax(norms)
    below_one = peak + np.argmax(norms[peak:] < 1)
    print(f'Peak of ||A^n||_2 = {norms[peak]:.2e} at n = {peak}')
    print(f'||A^n||_2 first drops below 1 at n = {below_one}')

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=n_range, y=norms, name="‖Aⁿ‖₂"))
    fig.add_trace(go.Scatter(x=n_range, y=rho ** n_range, name="ρ(A)ⁿ", line=dict(dash='dash')))
    fig.update_xaxes(title_text="n")
    fig.update_yaxes(type="log", exponentformat="power")
    fig.show()

part3()

rho(A) = 0.9, kappa(P) = 1.2e+299
Peak of ||A^n||_2 = 9.70e+17 at n = 189
||A^n||_2 first drops below 1 at n = 863


with k = 19. The polynomial n^19 grows much faster than 0.9^n shrinks until n = 189

cond(P) is useless here because the matrix is defective: the eigenvalue 0.9 is repeated 20 times but has only one eigenvector (e_1). So every column of P would be a multiple of e_1, P has rank 1, its smallest singular value is 0, and cond(P) = max/min comes out as infinite. NumPy reports cond =  10^299 only because of rounding. The bound cond(P)*p^n is then either inf or about 10^299·0.9^n. That’s technically true but says nothing: at the peak (n = 189) it is about 10^299 against a true value of about 10^18, and it only drops below 1 at n= 6500 instead of n = 863. It also has the wrong shape, since it decays from the start while the true norm first grows like n^19. For defective matrices, the right description is the Jordan-block growth n^k * |lambda|^n